In [1]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter

# 加载源数据文档

In [2]:
# Load the document, split it into chunks, embed each chunk and load it into the vector store.
raw_documents = TextLoader('./examples/rag.txt').load()
text_splitter = CharacterTextSplitter(separator='\n\n\n', chunk_size=50, chunk_overlap=4)
documents = text_splitter.split_documents(raw_documents)
documents[:2]

Created a chunk of size 109, which is longer than the specified 50
Created a chunk of size 65, which is longer than the specified 50
Created a chunk of size 143, which is longer than the specified 50
Created a chunk of size 833, which is longer than the specified 50
Created a chunk of size 263, which is longer than the specified 50
Created a chunk of size 304, which is longer than the specified 50
Created a chunk of size 609, which is longer than the specified 50
Created a chunk of size 174, which is longer than the specified 50
Created a chunk of size 449, which is longer than the specified 50
Created a chunk of size 280, which is longer than the specified 50
Created a chunk of size 325, which is longer than the specified 50
Created a chunk of size 560, which is longer than the specified 50
Created a chunk of size 318, which is longer than the specified 50
Created a chunk of size 68, which is longer than the specified 50


[Document(page_content='2024年普通高等学校招生全国统一考试（简称：2024年全国高考），是中华人民共和国合格的高中毕业生或具有同等学力的考生参加的选拔性考试 [1-2]。2024年报名人数1342万人，比2023年增加51万人 [21]。', metadata={'source': './examples/rag.txt'}),
 Document(page_content='2024年高考是黑龙江、甘肃、吉林、安徽、江西、贵州、广西7个省份（中国第四批高考综合改革省份）的第一届落地实施的新高考。 [3]', metadata={'source': './examples/rag.txt'})]

In [3]:
from llms.langchain_tutorial.deepseek import get_local_embeddings
from langchain_community.vectorstores import FAISS

db = FAISS.from_documents(documents, get_local_embeddings())

/opt/miniconda3/envs/langchain-stu/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# 相似度检索

In [4]:
query = "哪里可以了解高考成绩"
docs = db.similarity_search(query)
print(docs[0].page_content)

一、在哪里可以了解高考成绩、志愿填报时间和方式、各高校招生计划、往年录取参考等志愿填报权威信息？
各省级教育行政部门或招生考试机构官方网站、微信公众号等权威渠道都会公布今年高考各阶段工作时间安排，包括高考成绩公布时间和查询方式、志愿填报时间，以及今年各高校招生计划、往年录取情况参考等权威信息。考生和家长要及时关注本地官方权威渠道发布的消息内容。
考生高考志愿是高校录取的重要依据，请广大考生务必按照省级招生考试机构相关要求按时完成志愿填报。前期，教育部已会同有关部门协调互联网平台对省级招生考试机构和高校的官方网站、微信公众号等进行了权威标识，请广大考生在信息查询时认准官方权威渠道，切勿轻信网络不实信息。


In [ ]:
docs

# 通过向量的相似度检索

In [5]:
embedding_vector = get_local_embeddings().embed_query(query)
docs = db.similarity_search_by_vector(embedding_vector)
print(docs[0].page_content)

一、在哪里可以了解高考成绩、志愿填报时间和方式、各高校招生计划、往年录取参考等志愿填报权威信息？
各省级教育行政部门或招生考试机构官方网站、微信公众号等权威渠道都会公布今年高考各阶段工作时间安排，包括高考成绩公布时间和查询方式、志愿填报时间，以及今年各高校招生计划、往年录取情况参考等权威信息。考生和家长要及时关注本地官方权威渠道发布的消息内容。
考生高考志愿是高校录取的重要依据，请广大考生务必按照省级招生考试机构相关要求按时完成志愿填报。前期，教育部已会同有关部门协调互联网平台对省级招生考试机构和高校的官方网站、微信公众号等进行了权威标识，请广大考生在信息查询时认准官方权威渠道，切勿轻信网络不实信息。


# Maximum marginal relevance search (MMR)

In [6]:
query = "哪里可以了解高考成绩"
docs = db.max_marginal_relevance_search(query)
docs

[Document(page_content='一、在哪里可以了解高考成绩、志愿填报时间和方式、各高校招生计划、往年录取参考等志愿填报权威信息？\n各省级教育行政部门或招生考试机构官方网站、微信公众号等权威渠道都会公布今年高考各阶段工作时间安排，包括高考成绩公布时间和查询方式、志愿填报时间，以及今年各高校招生计划、往年录取情况参考等权威信息。考生和家长要及时关注本地官方权威渠道发布的消息内容。\n考生高考志愿是高校录取的重要依据，请广大考生务必按照省级招生考试机构相关要求按时完成志愿填报。前期，教育部已会同有关部门协调互联网平台对省级招生考试机构和高校的官方网站、微信公众号等进行了权威标识，请广大考生在信息查询时认准官方权威渠道，切勿轻信网络不实信息。', metadata={'source': './examples/rag.txt'}),
 Document(page_content='2024年高考是黑龙江、甘肃、吉林、安徽、江西、贵州、广西7个省份（中国第四批高考综合改革省份）的第一届落地实施的新高考。 [3]', metadata={'source': './examples/rag.txt'}),
 Document(page_content='报名人数\n2024年高考报名人数\n地区,报名情况\n全国,2024年全国高考报名人数达到1342万人，比去年增加了51万人。 [21]\n北京,2024年北京市报名参加高考考生共计6.72万人，全市共设18个考区、105个考点、2066个考场、315个备用考场。 [48]\n上海,约5.4万名考生参加考试。 [105]\n重庆,2024年，重庆市除高职分类考试招生、保送生等已录取考生和体育单招、残疾单招等考生外，应参加统一高考的考生20.3万人，比去年增加0.9万人。全市共设置考区39个，考点106个，考场6861个。 [74]', metadata={'source': './examples/rag.txt'}),
 Document(page_content='三、高校招生章程有什么作用，如何查询？\n高校招生章程由学校依据相关法律规定和国家招生政策制定，是学校开展招生工作的依据。考生在填报志愿前，应仔细查阅拟报考高校的招生章程，全面了解高校招生办法和相关招生要求。\n主要查询途径有：中国高等教育学生信

## 使用 DeepSeek 根据检索结果回答问题

In [7]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from llms.langchain_tutorial.deepseek import get_deepseek_chat

question = '哪里可以了解高考成绩？'
context = '\n\n'.join(doc.page_content for doc in db.similarity_search(question, k=3))
rag_chain = (
    ChatPromptTemplate.from_template('请仅根据资料回答问题；资料不足时说明不知道。\n资料：{context}\n问题：{question}')
    | get_deepseek_chat()
    | StrOutputParser()
)
print(rag_chain.invoke({'context': context, 'question': question}))

根据资料，可以了解高考成绩的渠道是：各省级教育行政部门或招生考试机构官方网站、微信公众号等权威渠道。
